# 📦 환경 설정 및 패키지 설치

LangGraph 기반 AI 에이전트를 구축하기 위해 필요한 패키지를 설치합니다.

---

In [ ]:
# ============================================================
# [패키지 설치 1] LangGraph 설치
# - LangGraph는 LangChain 생태계의 그래프 기반 에이전트 프레임워크입니다.
# - 노드(Node)와 엣지(Edge)로 구성된 그래프 구조로 복잡한 워크플로우를 표현합니다.
# 공식 문서: https://docs.langchain.com/oss/python/langgraph/install
# ============================================================
# !pip install langgraph

In [ ]:
# ============================================================
# [패키지 설치 2] LangChain 설치
# - LangChain은 LLM(대형 언어 모델) 기반 애플리케이션을 위한 프레임워크입니다.
# - 모델 연동, 도구(Tool), 메모리, 프롬프트 등의 기능을 제공합니다.
# ============================================================
# !pip install langchain

In [ ]:
# ============================================================
# [패키지 설치 3] Google Gemini 연동 패키지 설치 (선택 사항)
# - Google의 Gemini 모델을 LangChain에서 사용하기 위한 통합 패키지입니다.
# - OpenAI 대신 Gemini를 사용할 경우에만 설치합니다.
# 공식 문서: https://docs.langchain.com/oss/python/integrations/providers/overview
# ============================================================
# !pip install langchain-google-genai

In [4]:
# ============================================================
# [API 키 설정 방법 1] 환경 변수에 직접 API 키 입력 (간단한 방법)
#
# ⚠️ 주의: 실제 API 키를 코드에 직접 넣으면 보안상 위험합니다.
#    - GitHub 등 공개 저장소에 업로드하면 키가 노출됩니다.
#    - 강의 실습에서만 사용하고, 실제 프로젝트에서는 .env 파일을 사용하세요.
# ============================================================
import os

# OpenAI API 키 설정 (본인의 실제 API 키로 교체하세요)
# os.environ["OPENAI_API_KEY"] = "sk-proj-여기에_본인_API_키_입력"

In [1]:
# ============================================================
# [API 키 설정 방법 2] .env 파일을 사용하는 안전한 방법 (권장)
#
# 사용 방법:
#   1. 프로젝트 루트에 '.env' 파일을 생성합니다.
#   2. 파일 안에 다음 내용을 입력합니다:
#      OPENAI_API_KEY=sk-proj-여기에_본인_API_키_입력
#   3. .gitignore에 '.env'를 추가하여 Git 추적에서 제외합니다.
#
# load_dotenv()가 True를 반환하면 .env 파일을 성공적으로 불러온 것입니다.
# ============================================================
import os
from dotenv import load_dotenv
from openai import OpenAI

# .env 파일의 환경 변수를 현재 프로세스에 로드합니다.
# 반환값: True(성공) / False(실패 또는 .env 파일 없음)
load_dotenv()

False

---
# 🤖 Part 1. Agent 구축 기초

> **학습 목표:** LangGraph의 핵심 개념인 **State**, **Node**, **Edge**를 이해하고,
> 간단한 사칙연산 에이전트를 직접 구현합니다.

**LangGraph 기본 구성 요소:**
| 구성 요소 | 설명 | 비유 |
|---|---|---|
| **State** | 에이전트가 공유하는 데이터 구조 | 공유 메모장 |
| **Node** | 특정 작업을 수행하는 함수 | 담당 직원 |
| **Edge** | 노드 간 실행 흐름을 연결 | 업무 지시 경로 |

참고: https://docs.langchain.com/oss/python/langgraph/quickstart

---

## Step 1. Model 및 Tool 정의

에이전트가 사용할 **LLM 모델**과 **도구(Tool)**를 정의합니다.
- **모델**: 사용자의 요청을 이해하고 어떤 도구를 사용할지 결정합니다.
- **도구**: 실제 계산(곱셈, 덧셈, 나눗셈 등)을 수행하는 함수입니다.

In [149]:
# ============================================================
# LLM 모델 객체 생성
#
# init_chat_model(): LangChain의 통합 모델 초기화 함수
#   - provider 설정 없이 모델 이름만으로 자동 감지합니다.
#   - temperature=0: 항상 일관된(결정론적) 응답을 생성합니다.
#     (0에 가까울수록 일관성 ↑, 1에 가까울수록 창의성 ↑)
#
# 참고: https://docs.langchain.com/oss/python/integrations/providers/google
# ============================================================
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-4o-mini",    # 사용할 모델명 (OpenAI의 경량화 모델)
    temperature=0     # 응답의 일관성을 최대화 (창의성 없음)
)

# 모델 동작 확인 (주석 해제하여 테스트 가능)
# model.invoke("안녕하세요!")

In [2]:
# ============================================================
# 사칙연산 도구(Tool) 정의
#
# @tool 데코레이터:
#   - 일반 Python 함수를 LangChain이 인식하는 '도구'로 변환합니다.
#   - LLM이 함수의 이름과 docstring을 읽고 언제 사용할지 스스로 판단합니다.
#   - docstring의 Args 섹션이 매개변수 설명으로 LLM에게 전달됩니다.
#
# ⚠️ 중요: docstring을 영어로 작성하는 이유
#   - LLM에게 함수 설명이 직접 전달되므로 영어가 더 정확하게 이해됩니다.
# ============================================================
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b  # a와 b를 곱한 결과 반환


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b  # a와 b를 더한 결과 반환


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int (피제수)
        b: Second int (제수, 0이면 ZeroDivisionError 발생)
    """
    return a / b  # a를 b로 나눈 결과 반환

In [6]:
# ============================================================
# 도구 목록 생성
#
# 에이전트가 사용할 수 있는 모든 도구를 리스트로 관리합니다.
# 새로운 도구를 추가하려면 이 리스트에 추가하세요.
# ============================================================
tools = [multiply, add, divide]

In [7]:
# ============================================================
# 모델과 도구 바인딩 (연결)
#
# bind_tools(): 모델에게 "이런 도구들을 사용할 수 있다"고 알려줍니다.
#   - LLM은 사용자 요청을 분석하여 필요한 도구와 인자를 선택합니다.
#   - 실제 도구 실행은 별도의 tool_node에서 처리합니다.
#
# 이후 모델을 호출하면 직접 답변하거나, 도구 호출 요청을 반환합니다.
# ============================================================
model_with_tools = model.bind_tools(tools)

In [13]:
# 바인딩된 모델 객체 확인 (어떤 도구들이 연결되어 있는지 확인)
model_with_tools

## Step 2. State 정의

**State**는 에이전트 실행 중 모든 노드가 공유하는 **데이터 구조**입니다.

```
[사용자] → 메시지 전달 → [에이전트]
                         ↕ State (공유 메모리)
                    [LLM 노드] ↔ [도구 노드]
```

- `messages`: 대화 기록을 저장 (누적 방식)
- `llm_calls`: LLM 호출 횟수 추적 (비용 모니터링용)

In [14]:
# ============================================================
# State 클래스 정의
#
# TypedDict: 타입 힌트가 있는 딕셔너리 (Python 3.8+)
#   - 각 키의 타입을 명시하여 코드 안정성을 높입니다.
#
# Annotated[list[AnyMessage], operator.add]:
#   - 'messages' 키에 새 메시지가 올 때 기존 목록에 '추가(append)'합니다.
#   - operator.add는 두 리스트를 합치는 방식 (덮어쓰기가 아님!)을 정의합니다.
#   - 예: 기존=[msg1, msg2] + 새로운=[msg3] → [msg1, msg2, msg3]
#
# add_messages 함수를 사용해도 동일한 효과를 얻을 수 있습니다.
# 참고: https://reference.langchain.com/python/langgraph
# ============================================================
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator

class MessagesState(TypedDict):
    # 대화 메시지 목록: 새 메시지는 기존 목록에 추가됨
    messages: Annotated[list[AnyMessage], operator.add]
    
    # LLM 호출 횟수 카운터: 비용 추적 및 무한 루프 방지에 활용
    llm_calls: int

### 📌 참고: add_messages 함수

`Annotated[list[AnyMessage], operator.add]` 대신 `add_messages`를 사용할 수도 있습니다.  
두 방식의 차이점은 [공식 문서](https://reference.langchain.com/python/langgraph/graphs/?_gl=1*ry73wr*_gcl_au*MTQ0NjcwMDQyNS4xNzYxNzg2NzMz*_ga*MjEyNDg4NjgwNC4xNzYyOTI0NjMz*_ga_47WX3HKKY2*czE3NjgyNzQ0NjEkbzI4JGcxJHQxNzY4Mjc2NzA5JGozMyRsMCRoMA..#langgraph.graph.message.add_messages)를 참고하세요.

## Step 3. Node 정의

**Node**는 그래프에서 실제 작업을 수행하는 함수입니다.

이 에이전트는 두 가지 노드로 구성됩니다:
1. **`llm_call`**: LLM을 호출하여 응답을 생성하거나 도구 사용을 요청
2. **`tool_node`**: LLM이 요청한 도구를 실제로 실행

In [15]:
# ============================================================
# [Node 1] llm_call - LLM 호출 노드
#
# 이 노드의 역할:
#   1. 현재 대화 기록(state["messages"])을 LLM에게 전달
#   2. LLM이 직접 답변하거나, 도구 사용을 요청하는 응답 반환
#
# 반환 형식:
#   - {"messages": [응답 메시지], "llm_calls": 호출횟수+1}
#   - State의 messages에 새 응답이 추가됨 (operator.add 방식)
# ============================================================
from langchain.messages import SystemMessage

def llm_call(state):
    """LLM이 현재 상태를 보고 답변하거나, 도구 사용을 요청하는 Node"""
    
    # SystemMessage: LLM에게 역할과 지침을 부여하는 메시지
    # state["messages"]: 지금까지의 대화 기록 (사용자 질문 포함)
    response = model_with_tools.invoke(
        [
            SystemMessage(
                content="당신은 사칙연산을 하는 유능한 Agent입니다."
            )
        ] + state["messages"]  # 시스템 메시지 + 기존 대화 기록 결합
    )
    
    # 변경된 상태 반환 (LangGraph가 자동으로 State에 병합)
    return {
        "messages": [response],                          # 새 응답을 메시지 목록에 추가
        "llm_calls": state.get('llm_calls', 0) + 1      # 호출 횟수 1 증가
    }

In [17]:
# ============================================================
# 도구 이름으로 빠르게 접근하기 위한 딕셔너리 생성
#
# {"multiply": <함수>, "add": <함수>, "divide": <함수>} 형태
# tool_node에서 LLM이 요청한 도구를 이름으로 찾아 실행할 때 사용합니다.
# ============================================================
tools_by_name = {tool.name: tool for tool in tools}

In [18]:
# 도구 딕셔너리 내용 확인 (각 도구의 이름, 설명, 스키마 포함)
tools_by_name

In [19]:
# ============================================================
# [Node 2] tool_node - 도구 실행 노드
#
# 이 노드의 역할:
#   - LLM이 도구 사용을 요청했을 때(tool_calls가 있을 때) 실제 함수를 실행
#
# 처리 흐름:
#   1. 가장 최근 LLM 응답에서 tool_calls 목록을 꺼냄
#   2. 각 tool_call에서 함수 이름과 인자를 추출
#   3. 해당 함수를 실행하여 결과를 얻음
#   4. 결과를 ToolMessage로 포장하여 State에 추가
#
# ToolMessage 구조:
#   - content: 도구 실행 결과 (문자열)
#   - tool_call_id: LLM 요청과 결과를 매칭하는 고유 ID (필수!)
# ============================================================
from langchain.messages import ToolMessage

def tool_node(state):
    """LLM이 도구 사용을 요청했을 때, 실제로 도구를 실행하는 단계"""
    
    result = []
    
    # 가장 최근 메시지(LLM 응답)에서 도구 호출 목록을 순회
    for tool_call in state["messages"][-1].tool_calls:
        
        # 1단계: 도구 이름으로 실제 함수 찾기
        #   예: tool_call["name"] = "multiply" → tools_by_name["multiply"] 함수
        tool = tools_by_name[tool_call["name"]]
        
        # 2단계: 도구(함수)를 LLM이 제공한 인자로 실행
        #   예: tool_call["args"] = {"a": 3, "b": 4} → multiply(a=3, b=4)
        tool_result = tool.invoke(tool_call["args"])
        
        # 3단계: 결과를 ToolMessage로 포장
        #   tool_call_id는 LLM이 어떤 요청에 대한 결과인지 식별하기 위해 필수입니다.
        result.append(
            ToolMessage(
                content=tool_result,           # 계산 결과
                tool_call_id=tool_call["id"]   # 요청-응답 매칭 ID
            )
        )
        
    # 실행 결과를 대화 기록(State)에 추가
    return {"messages": result}

In [ ]:
# ============================================================
# 📋 에이전트 실행 예시: "42 + 3 * 23은 뭔가요?"
#
# 아래는 실제 에이전트가 이 질문을 처리하는 메시지 흐름입니다:
#
# [1] HumanMessage: 사용자 질문 입력
# [2] AIMessage (tool_calls=[multiply]): LLM이 먼저 3*23 계산 요청
# [3] ToolMessage (69): multiply 도구 실행 결과
# [4] AIMessage (tool_calls=[add]): LLM이 42+69 계산 요청
# [5] ToolMessage (111): add 도구 실행 결과
# [6] AIMessage: 최종 답변 "정답은 111입니다"
#
# ⚠️ 참고: 연산 순서(곱셈 먼저)를 LLM이 스스로 판단합니다!
# ============================================================
{'messages': [
    # 사용자의 질문
    # HumanMessage(content='42 + 3 * 23은 뭔가요?', ...),
    
    # LLM이 곱셈부터 하기로 결정 (tool_calls 포함)
    # AIMessage(content='', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 23}}]),
    
    # 곱셈 결과
    # ToolMessage(content='69', name='multiply'),
    
    # LLM이 덧셈 계산 요청
    # AIMessage(content='', tool_calls=[{'name': 'add', 'args': {'a': 42, 'b': 69}}]),
    
    # 덧셈 결과
    # ToolMessage(content='111', name='add'),
    
    # 최종 답변
    # AIMessage(content='정답은 111입니다. (42 + 3*23 = 42 + 69 = 111)')
    ]
}

## Step 4. 그래프(Graph) 생성

노드와 엣지를 연결하여 **에이전트 실행 흐름**을 정의합니다.

```
START → llm_call → (도구 필요?) → tool_node → llm_call (반복)
                 ↓ (도구 불필요)
                END
```

In [20]:
# ============================================================
# 그래프 빌더 생성 및 노드/엣지 추가
#
# StateGraph: State를 공유하는 그래프 구조
#   - MessagesState 타입을 사용하여 State 스키마를 지정합니다.
#
# add_node(이름, 함수): 그래프에 노드를 추가
# add_edge(출발, 도착): 무조건 실행되는 연결선 추가
#   - START: 그래프 실행의 시작점 (특수 상수)
#   - "tool_node" → "llm_call": 도구 실행 후 다시 LLM 노드로 돌아옴
# ============================================================
from langgraph.graph import StateGraph, START

# 1. 그래프 빌더 생성 (State 스키마 지정)
agent_builder = StateGraph(MessagesState)

# 2. 노드 등록
agent_builder.add_node("llm_call", llm_call)   # LLM 호출 노드
agent_builder.add_node("tool_node", tool_node)  # 도구 실행 노드

# 3. 고정 엣지 연결
agent_builder.add_edge(START, "llm_call")       # 시작 → LLM 호출
agent_builder.add_edge("tool_node", "llm_call") # 도구 실행 후 → LLM 다시 호출

In [21]:
# ============================================================
# 조건부 엣지(Conditional Edge) 함수 정의
#
# 역할: LLM 응답을 분석하여 다음 실행할 노드를 결정합니다.
#
# 판단 기준:
#   - LLM이 tool_calls를 포함하면 → "tool_node"로 이동 (도구 실행)
#   - tool_calls가 없으면 → END (대화 종료)
#
# 반환값은 반드시 노드 이름(문자열) 또는 END 상수여야 합니다.
# ============================================================
from langgraph.graph import END

def should_continue(state: MessagesState):
    """다음 단계를 결정하는 라우팅 함수"""
    last_message = state['messages'][-1]  # 가장 최근 메시지 확인
    
    if last_message.tool_calls:
        # LLM이 도구 사용을 요청한 경우
        return "tool_node"
    else:
        # LLM이 최종 답변을 완성한 경우 (도구 요청 없음)
        return END

In [22]:
# ============================================================
# 조건부 엣지 연결
#
# add_conditional_edges(출발 노드, 조건 함수, 가능한 목적지 목록)
#
# 동작 방식:
#   1. "llm_call" 노드 실행 완료
#   2. should_continue() 함수 호출하여 반환값 확인
#   3. 반환값에 따라 다음 노드로 이동:
#      - "tool_node" → tool_node 노드 실행
#      - END → 그래프 종료
# ============================================================
agent_builder.add_conditional_edges(
    'llm_call',        # 조건부 엣지의 출발 노드
    should_continue,   # 다음 노드를 결정하는 함수
    [END, 'tool_node'] # 가능한 목적지 목록 (자동 유효성 검사)
)

In [23]:
# ============================================================
# 그래프 컴파일 (에이전트 완성)
#
# compile(): 그래프 정의를 실행 가능한 에이전트로 변환합니다.
#   - 노드와 엣지 유효성 검사
#   - 실행 계획 최적화
#   - invoke(), stream() 등의 메서드 활성화
#
# 선택적 옵션:
#   - checkpointer: 대화 기록 저장소 (멀티턴 대화에 필요)
#   - interrupt_before/after: 특정 노드 전후에 실행을 일시 중지
# ============================================================
agent = agent_builder.compile()

In [24]:
# 컴파일된 에이전트 그래프 시각화
# 노드(사각형)와 엣지(화살표)로 구성된 실행 흐름을 확인합니다.
agent

## Step 5. 에이전트 실행 (기본 테스트)

In [25]:
# ============================================================
# 에이전트 실행 - 단순 덧셈
#
# invoke(): 에이전트를 동기적으로 실행하고 최종 결과를 반환합니다.
# 입력 형식: {"messages": [메시지 목록]}
#
# 예상 실행 흐름:
#   START → llm_call("3과 4를 더해줘") → tool_calls=[add(3,4)]
#         → tool_node(add 실행 → 7) → llm_call(최종 답변) → END
# ============================================================
from langchain.messages import HumanMessage

# 사용자 질문 설정
messages = [HumanMessage(content="3과 4를 더해줘")]

# 에이전트 실행
response = agent.invoke({"messages": messages})

In [26]:
# 전체 에이전트 응답 확인
# - messages: 전체 대화 기록 (사용자 질문 → AI 응답 → 도구 결과 → 최종 답변)
# - llm_calls: LLM 호출 횟수 (이 예제에서는 2회: 도구 요청 + 최종 답변)
response

In [38]:
# ============================================================
# 최종 답변만 추출
#
# response["messages"][-1]: 대화 기록의 마지막 메시지 (= 최종 AI 답변)
# .content: 메시지의 텍스트 내용
# ============================================================
response["messages"][-1].content

In [ ]:
# Gemini 모델을 사용하는 경우 응답 출력 방법
# (Gemini 응답은 content가 리스트 형태로 반환될 수 있음)
# print(response["messages"][-1].content[0]['text'])

In [39]:
# ============================================================
# 에이전트 실행 - 복합 연산 (덧셈 후 곱셈)
#
# 예상 실행 흐름 (2단계 도구 사용):
#   1. llm_call: "3+4를 먼저 계산" → tool_calls=[add(3,4)]
#   2. tool_node: add 실행 → 7
#   3. llm_call: "7*7을 계산" → tool_calls=[multiply(7,7)]
#   4. tool_node: multiply 실행 → 49
#   5. llm_call: "최종 답변: 49" → END
# ============================================================
messages = [HumanMessage(content="3과 4를 더한 뒤 7을 곱해줘.")]
response = agent.invoke({"messages": messages})

In [41]:
# 복합 연산 결과 확인 (예상: "49입니다")
response["messages"][-1].content

---
## 🔄 OpenAI 기반 에이전트로 재구성

위에서 만든 에이전트를 OpenAI 최신 모델(gpt-5-nano)로 재구성합니다.

> **왜 모델을 바꾸는가?**  
> - 더 강력한 추론 능력을 활용하기 위해  
> - 다양한 모델로 전환하는 방법을 익히기 위해

---

In [ ]:
# OpenAI 전용 LangChain 패키지 설치 (미설치 시)
!pip install -qU langchain-openai

In [ ]:
# OpenAI API 키 설정 (환경 변수에 없는 경우만 사용)
# import os
# os.environ["OPENAI_API_KEY"] = "sk-proj-여기에_본인_API_키_입력"

In [42]:
# ============================================================
# 새 모델로 교체: gpt-5-nano
#
# 참고: temperature 미지정 시 기본값(1.0)이 적용됩니다.
#   - 사칙연산처럼 정답이 명확한 작업에는 temperature=0이 권장됩니다.
# ============================================================
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

In [43]:
# 새 모델에 동일한 도구 목록 바인딩
model_with_tools = model.bind_tools(tools)

In [44]:
# ============================================================
# [Node 1 재정의] llm_call - 새 모델로 업데이트
#
# 이전과 동일한 로직이지만, 새로 정의한 model_with_tools를 사용합니다.
# 함수 재정의 시 LangGraph 그래프에도 자동 반영됩니다.
# ============================================================
from langchain.messages import SystemMessage

def llm_call(state):
    """LLM이 현재 상태를 보고 답변하거나, 도구 사용을 요청하는 Node"""

    # 시스템 메시지(역할 지정) + 기존 대화 기록을 합쳐서 LLM에 전송
    response = model_with_tools.invoke(
        [
            SystemMessage(
                content="당신은 사칙연산을 하는 유능한 Agent입니다."
            )
        ] + state["messages"]
    )

    # 변경된 상태를 반환 (messages에 새 응답 추가, llm_calls 1 증가)
    return {
        "messages": [response],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

In [45]:
# 도구 이름-함수 매핑 딕셔너리 재생성 (이전과 동일)
tools_by_name = {tool.name: tool for tool in tools}

In [46]:
# ============================================================
# [Node 2 재정의] tool_node - 도구 실행 노드 (이전과 동일한 로직)
# ============================================================
from langchain.messages import ToolMessage

def tool_node(state):
    """LLM이 도구 사용을 요청했을 때, 실제로 도구를 실행하는 단계"""

    result = []
    
    # 가장 최근 메시지(LLM의 응답)에서 도구 호출 요청(tool_calls)들을 꺼내기
    for tool_call in state["messages"][-1].tool_calls:
        
        # 1단계: 도구 이름으로 실제 함수 찾기
        tool = tools_by_name[tool_call["name"]]

        # 2단계: 함수를 실행하여 결과를 얻기
        tool_result = tool.invoke(tool_call["args"])

        # 3단계: 결과를 ToolMessage 형태로 포장 (tool_call_id는 필수!)
        result.append(
            ToolMessage(
                content=tool_result,
                tool_call_id=tool_call["id"]
            )
        )

    # 실행 결과를 대화 기록에 추가
    return {"messages": result}

In [48]:
# ============================================================
# 그래프 재구성 (새 모델 적용)
#
# 1. 워크플로우(그래프) 생성
# 2. 노드(작업자) 배치: llm_call, tool_node
# 3. 엣지(연결선) 연결: START → llm_call, tool_node → llm_call
# ============================================================
from langgraph.graph import StateGraph, START

# 1. 새 워크플로우(그래프) 생성
agent_builder = StateGraph(MessagesState)

# 2. 노드(작업자) 배치
agent_builder.add_node("llm_call", llm_call)   # LLM 호출 노드
agent_builder.add_node("tool_node", tool_node)  # 도구 실행 노드

# 3. 엣지(연결선) 연결
agent_builder.add_edge(START, "llm_call")       # 시작 → LLM 호출
agent_builder.add_edge("tool_node", "llm_call") # 도구 실행 후 → LLM 다시 호출

In [50]:
# ============================================================
# 조건부 라우팅 함수 정의 및 엣지 연결
#
# should_continue(): LLM 응답에 tool_calls가 있는지 확인
#   - 있으면 → "tool_node" (도구 실행 계속)
#   - 없으면 → END (최종 답변 완료)
# ============================================================
from langgraph.graph import StateGraph, START, END

def should_continue(state: MessagesState):
    """
    LLM의 응답을 보고 다음 단계로 어디를 갈지 결정
    """
    messages = state["messages"]
    last_message = messages[-1]  # 가장 최근 메시지

    # LLM이 도구 호출(tool_calls)을 포함한 응답을 보낸 경우
    if last_message.tool_calls:
        return "tool_node"  # 도구 실행 노드로 이동

    # 도구 호출이 없으면 작업 종료 (최종 답변 완성)
    return END

In [51]:
# ============================================================
# 조건부 엣지 연결
#
# 인자 설명:
#   - "llm_call": 조건부 엣지가 시작되는 노드
#   - should_continue: 다음 노드를 결정하는 라우팅 함수
#   - ["tool_node", END]: 가능한 목적지 (유효성 자동 검사)
# ============================================================
agent_builder.add_conditional_edges(
    "llm_call",          # 출발지 노드
    should_continue,     # 판단 로직 함수
    ["tool_node", END]   # 갈 수 있는 목적지들
)

In [52]:
# 컴파일 (최종 에이전트 생성)
agent = agent_builder.compile()

In [53]:
# 새 에이전트 실행 테스트
from langchain.messages import HumanMessage

messages = [HumanMessage(content="3과 4를 더해줘")]
response = agent.invoke({"messages": messages})

In [54]:
# 전체 응답 확인 (메시지 기록 + llm_calls 횟수)
response

In [ ]:
# ============================================================
# 📋 참고: Gemini 모델 사용 시 응답 형식
#
# Gemini 응답의 특징:
#   - content가 문자열이 아닌 리스트 형태로 반환됩니다.
#   - 예: content=[{'type': 'text', 'text': '3과 4를 더하면 7입니다.'}]
#   - 최종 답변 추출 시: response["messages"][-1].content[0]['text']
#
# 아래는 Gemini 기반 에이전트 실행 시 실제 response 예시입니다.
# ============================================================
# Gemini 기반 Agent 실행 시 response 결과

---
# 🏭 Part 2. Agent 구축 실전 1 - 이메일 처리 에이전트

> **학습 목표:**  
> - **Router 패턴** 구현: 조건에 따라 다른 처리 경로로 분기
> - 규칙 기반(keyword) 라우팅으로 빠르고 예측 가능한 분기 처리

**시나리오:** 고객 이메일을 자동으로 분류하고 처리하는 에이전트
- 환불/긴급 키워드 → 상담원 이관
- 일반 문의 → 매뉴얼 검색 후 자동 답변

```
START → read_email → classify_intent → (라우팅)
                                        ├─ complaint → escalate_to_human → END
                                        └─ inquiry → search_manual → write_reply → END
```

참고: https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph

---

## Step 1. Model 정의 (선택)

In [ ]:
# ============================================================
# Google Gemini 모델 사용 시 (선택 사항)
#
# from langchain_google_genai import ChatGoogleGenerativeAI
# model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
# ============================================================
# OpenAI 모델은 이미 위에서 초기화했으므로 그대로 사용합니다.

## Step 2. State 정의

이메일 처리에 특화된 State 구조를 정의합니다.

| 키 | 타입 | 설명 |
|---|---|---|
| `email_content` | str | 처리할 이메일 원본 내용 |
| `category` | str | 분류 결과 ('complaint' 또는 'inquiry') |
| `next_step` | str | 다음 실행할 단계 이름 |
| `response` | str | 최종 AI 답변 |

In [55]:
# ============================================================
# 이메일 처리 에이전트의 State 정의
#
# 이 State는 이메일 처리 파이프라인 전체에서 공유됩니다.
# 각 노드는 필요한 필드를 읽고, 처리 결과를 해당 필드에 저장합니다.
# ============================================================
from typing_extensions import TypedDict

class AgentState(TypedDict):
    email_content: str  # 처리할 이메일 원본 내용
    category: str       # 이메일 분류 결과 ('complaint' 또는 'inquiry')
    next_step: str      # 라우팅에 사용할 다음 처리 단계
    response: str       # 최종 AI 생성 답변

## Step 3. Node 정의

이메일 처리 파이프라인의 각 단계를 노드로 구현합니다.

In [56]:
# ============================================================
# [Node 1] read_email - 이메일 읽기 노드
#
# 역할: 이메일 내용을 State에서 읽어 그대로 반환
# 현재는 단순 패스스루(pass-through) 역할이지만,
# 실제 프로젝트에서는 이메일 파싱, 전처리 등이 여기서 이루어집니다.
# ============================================================
def read_email(state: AgentState):
    """이메일 내용을 읽어 State에 저장합니다."""
    return {'email_content': state['email_content']}

In [57]:
# ============================================================
# [Node 2] classify_intent - 의도 분류 노드 (Router의 핵심)
#
# 역할: 이메일 내용을 분석하여 처리 경로를 결정합니다.
#
# 분류 기준 (규칙 기반 - 키워드 매칭):
#   - '환불', '긴급', '빨리' 포함 → complaint (불만) → 상담원 이관
#   - 그 외 → inquiry (일반 문의) → 자동 답변
#
# ⚠️ 실전에서는 LLM 기반 분류(더 정확)를 사용하는 것이 좋습니다.
#    (이 방식은 키워드 목록을 계속 업데이트해야 하는 단점이 있습니다.)
# ============================================================
def classify_intent(state: AgentState):
    """이메일의 의도를 분류하고 다음 처리 단계를 결정합니다."""
    email = state['email_content']
    
    # 키워드 기반 분류
    if '환불' in email or '긴급' in email or '빨리' in email:
        category = 'complaint'          # 불만/긴급 이메일
        next_step = 'escalate_to_human' # → 상담원에게 이관
    else:
        category = 'inquiry'            # 일반 문의 이메일
        next_step = 'search_manual'     # → 매뉴얼 검색 후 자동 답변
        
    # 분류 결과를 State에 저장
    return {'category': category, 'next_step': next_step}

In [58]:
# ============================================================
# [Node 3-A] search_manual - 매뉴얼 검색 노드 (자동 처리 트랙)
#
# 역할: 일반 문의에 대해 관련 매뉴얼/FAQ를 검색합니다.
# 현재는 로그 출력만 하지만, 실제로는 RAG(검색 증강 생성) 시스템과 연동됩니다.
# ============================================================
def search_manual(state: AgentState):
    """관련 매뉴얼이나 FAQ를 검색합니다."""
    print('3-A 진입... 매뉴얼을 검색합니다')
    return  # 실제 프로젝트에서는 검색 결과를 State에 추가

# ============================================================
# [Node 3-B] escalate_to_human - 상담원 이관 노드 (긴급 이슈 트랙)
#
# 역할: 불만/긴급 이메일을 사람 상담원에게 이관합니다.
# 실제로는 CRM 시스템 API 호출, 슬랙 알림, 티켓 생성 등이 여기서 처리됩니다.
# ============================================================
def escalate_to_human(state: AgentState):
    """이메일을 전문 상담원에게 이관합니다."""
    print('3-B 진입... 상담원 이관합니다.')
    return

In [59]:
# ============================================================
# [Node 4] write_reply - 답변 생성 노드
#
# 역할: 이메일 내용을 기반으로 AI가 답변을 생성합니다.
# 매뉴얼 검색 결과를 활용하면 더 정확한 답변이 가능합니다.
# ============================================================
def write_reply(state: AgentState):
    """이메일에 대한 AI 답변을 생성합니다."""
    email = state['email_content']
    
    # 이메일 내용을 LLM에 전달하여 답변 생성
    # 실제 프로젝트에서는 매뉴얼 검색 결과도 함께 전달합니다.
    response = model.invoke(email)
    
    return {'response': response}  # 생성된 답변을 State에 저장

## Step 4. 그래프 생성

In [60]:
# ============================================================
# 이메일 처리 에이전트 그래프 구성
#
# 흐름 설계:
#   START → read_email → classify_intent → (조건부 분기)
#                                           ├─ complaint → escalate_to_human → END
#                                           └─ inquiry → search_manual → write_reply → END
# ============================================================
from langgraph.graph import StateGraph, START, END

# 1. 워크플로우(그래프) 생성
agent_builder = StateGraph(AgentState)

# 2. 노드(작업자) 배치 - 순서에 상관없이 모두 등록
agent_builder.add_node('read_email', read_email)               # 이메일 읽기
agent_builder.add_node('classify_intent', classify_intent)     # 의도 분류
agent_builder.add_node('search_manual', search_manual)         # 매뉴얼 검색
agent_builder.add_node('escalate_to_human', escalate_to_human) # 상담원 이관
agent_builder.add_node('write_reply', write_reply)             # 답변 생성

# 3. 고정 엣지 연결 (무조건 실행되는 경로)
agent_builder.add_edge(START, 'read_email')              # 시작 → 이메일 읽기
agent_builder.add_edge('read_email', 'classify_intent')  # 읽기 → 분류

In [61]:
# ============================================================
# 라우팅 함수 정의
#
# State의 next_step 값을 기반으로 다음 노드를 결정합니다.
# classify_intent 노드가 저장한 값을 읽어 분기 방향을 결정합니다.
#
# 반환값: 이동할 노드의 이름 (문자열)
# ============================================================
def route_email(state: AgentState):
    """이메일 분류 결과에 따라 처리 경로를 결정합니다."""
    if state['next_step'] == 'escalate_to_human':
        return 'escalate_to_human'  # 상담원 이관 트랙
    else:
        return 'search_manual'      # 자동 답변 트랙

In [62]:
# ============================================================
# 조건부 엣지 연결
#
# classify_intent 노드 실행 후 route_email 함수를 호출하여
# 결과에 따라 다음 노드를 결정합니다.
# ============================================================
agent_builder.add_conditional_edges(
    'classify_intent',  # 출발 노드 (분류 완료 후 분기)
    route_email,        # 분기 결정 함수
    ['escalate_to_human', 'search_manual']  # 가능한 목적지
)

In [63]:
# ============================================================
# 나머지 엣지 연결
#
# 트랙 A (자동 처리): search_manual → write_reply → END
# 트랙 B (이관): escalate_to_human → END
# ============================================================
agent_builder.add_edge('search_manual', 'write_reply')  # 검색 → 답변 작성
agent_builder.add_edge('write_reply', END)              # 답변 완료 → 종료
agent_builder.add_edge('escalate_to_human', END)        # 이관 완료 → 종료

In [64]:
# 그래프 컴파일 (최종 에이전트 생성)
agent = agent_builder.compile()

In [65]:
# 이메일 처리 에이전트 그래프 시각화 (분기 구조 확인)
agent

## Step 5. 에이전트 실행 테스트

In [66]:
# ============================================================
# 테스트 1: 일반 문의 이메일 처리
#
# 예상 흐름:
#   read_email → classify_intent('inquiry') → search_manual → write_reply → END
#
# 콘솔 출력: "3-A 진입... 매뉴얼을 검색합니다"
# ============================================================
inputs = {"email_content": "비밀번호 변경 방법을 알려주세요."}
response = agent.invoke(inputs)

In [67]:
# 일반 문의 처리 결과 확인
# - category: 'inquiry' (일반 문의로 분류됨)
# - next_step: 'search_manual' (매뉴얼 검색으로 라우팅)
# - response: AI가 생성한 답변 (AIMessage 객체)
response

In [70]:
# AI 생성 답변의 텍스트 내용만 출력
print(response['response'].content)

In [71]:
# ============================================================
# 테스트 2: 환불 요청 이메일 처리
#
# 예상 흐름:
#   read_email → classify_intent('complaint') → escalate_to_human → END
#
# '환불' 키워드 감지 → 상담원 이관 경로 선택
# 콘솔 출력: "3-B 진입... 상담원 이관합니다."
# ============================================================
inputs = {"email_content": "당장 환불해줘!"}
response = agent.invoke(inputs)

In [72]:
# 환불 요청 처리 결과 확인
# - category: 'complaint' (불만으로 분류됨)
# - next_step: 'escalate_to_human' (상담원 이관으로 라우팅)
# - response 키 없음: 자동 답변 없이 이관만 처리
response

In [73]:
# ============================================================
# 테스트 3: '환급' 키워드 테스트
#
# '환급'은 키워드 목록에 없으므로 일반 문의로 분류됩니다.
# 이처럼 규칙 기반 분류는 키워드 관리가 중요합니다.
# → 해결책: LLM 기반 분류 (다음 실전 예제에서 구현)
# ============================================================
inputs = {"email_content": "당장 환급해줘!"}
response = agent.invoke(inputs)

In [74]:
# '환급' 처리 결과 - 일반 문의로 처리됨 (키워드 미매칭)
# 이 결과를 보고 '환급'도 키워드에 추가할지 검토해야 합니다.
response

---
# 🏭 Part 3. Agent 구축 실전 2 - LLM 기반 이메일 분류 에이전트

> **학습 목표:**  
> - 규칙 기반 분류의 한계를 극복하기 위해 **LLM 기반 분류** 구현
> - **다중 노드 그래프**: 분류 → 상담/이관 → 도구 사용 패턴 습득
> - 이전 예제와의 차이: 분류도 LLM이 담당하므로 더 유연하고 정확함

**구조 개선 포인트:**
- 이전: 키워드 매칭(규칙 기반) → 경직된 분류
- 이번: LLM 판단(AI 기반) → 유연하고 정확한 분류

```
START → classify_node(LLM 판단)
            ├─ escalate → escalate_node → END
            └─ consultant → consultant_node(도구 사용)
                              ├─ tool_calls 있음 → tool_node → consultant_node (반복)
                              └─ tool_calls 없음 → END
```

---

## Step 1. Model 및 Tool 정의

In [174]:
# ============================================================
# 모델 초기화
#
# temperature=0: 분류 작업은 일관된 결과가 중요하므로 0 설정
# (LLM 기반 분류에서 매번 다른 결과가 나오면 신뢰성이 낮아집니다)
# ============================================================
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-4o-mini",
    temperature=0  # 일관성 있는 분류를 위해 0 설정
)

In [175]:
# ============================================================
# 매뉴얼 검색 도구 정의
#
# 이 도구는 고객 문의에 대한 답변을 찾기 위해 사용됩니다.
# 실제 프로젝트에서는 벡터 DB나 검색 엔진과 연동됩니다.
#
# 현재는 시뮬레이션용 더미 데이터를 반환합니다.
# ============================================================
from langchain.tools import tool

@tool
def search_manual(query: str) -> str:
    """
    고객의 질문에 답변하기 위해 참고할만한 규정이나 매뉴얼을 검색할 때 사용하는 도구.
    """
    # 키워드 기반 더미 검색 (실제 구현 시 RAG 시스템으로 교체)
    if '비밀번호' in query:
        return '비밀번호 변경은 마이페이지 - 보안 설정에 있음'
    elif '배송' in query:
        return '00택배에서 3일 내 배송 예정임'
    else:
        return '해당 내용 관련 매뉴얼은 찾을 수 없습니다.'

In [176]:
# 도구 목록 설정 (현재는 매뉴얼 검색 도구만 사용)
tools = [search_manual]

In [177]:
# 상담 AI에 검색 도구 바인딩
# 상담 노드(consultant_node)만 도구를 사용합니다.
# 분류 노드(classify_node)는 도구 없이 순수 LLM만 사용합니다.
model_with_tools = model.bind_tools(tools)

## Step 2. State 정의

In [178]:
# ============================================================
# 개선된 State 정의
#
# 이전 예제와의 차이점:
#   - messages: 대화 형식으로 관리 (add_messages 방식)
#   - next_step: 라우팅 결정을 위한 플래그
#
# add_messages vs operator.add:
#   - 기능적으로 유사하지만 add_messages는 메시지 ID 중복 처리 등
#     추가적인 안전 장치를 제공합니다.
# ============================================================
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    # 전체 대화 메시지 기록 (새 메시지는 누적 방식으로 추가)
    messages: Annotated[list[AnyMessage], operator.add]
    
    # 분류 결과에 따른 다음 처리 단계
    # 'consultant' (자동 상담) 또는 'escalate' (상담원 이관)
    next_step: str

## Step 3. Node 정의

In [179]:
# ============================================================
# [Node 1] classify_node - LLM 기반 분류 노드 (Manager 역할)
#
# 이전 예제의 classify_intent와 다른 점:
#   - 키워드 매칭(규칙) 대신 LLM이 이메일 의도를 직접 판단
#   - 더 자연스러운 표현도 정확하게 분류 가능 ('환급', '급해요' 등)
#
# 구현 포인트:
#   - System Prompt로 LLM의 역할과 출력 형식을 제한
#   - "단어 하나만 반환" → 파싱이 간단해짐
#   - Gemini 등 응답 형식이 다른 모델에 대한 호환성 처리 포함
# ============================================================
from langchain.messages import SystemMessage

def classify_node(state: AgentState):
    """LLM을 사용하여 이메일 의도를 분류하는 Manager 노드"""
    print("\n--- [1] 분류 단계 (LLM 판단) ---")
    
    # 상태 디버깅용 전역 변수 (선택 사항)
    global global_state
    global_state = state
    
    # 메시지 유효성 검사
    messages = state.get("messages", [])
    if not messages:
        raise ValueError("상태에 메시지가 없습니다.")
        
    last_message = state["messages"][-1]

    # --------------------------------------------------------
    # 분류용 System Prompt
    # LLM에게 역할과 출력 형식을 명확히 지정합니다.
    # "단어 하나만" 제약으로 파싱 오류를 방지합니다.
    # --------------------------------------------------------
    prompt = """
    당신은 고객 센터 관리자입니다. 고객의 이메일을 분석해서 다음 단계를 결정하세요.

    1. 단순 문의나 정보 요청이라면 -> 'consultant' 반환
    2. 환불 요청, 불만 제기, 화난 고객이라면 -> 'escalate' 반환

    답변은 오직 단어 하나만 하세요.
    """

    # LLM 호출 (도구 없는 순수 모델 사용)
    response = model.invoke([SystemMessage(content=prompt)] + [last_message])
    raw_content = response.content

    # --------------------------------------------------------
    # 모델별 응답 형식 처리
    # OpenAI: content가 문자열
    # Gemini: content가 텍스트 블록 리스트
    # --------------------------------------------------------
    if isinstance(raw_content, list):
        # Gemini 등 리스트 형태로 오는 경우
        decision = "".join([block['text'] for block in raw_content if block.get('type') == 'text'])
    else:
        # OpenAI 등 문자열로 오는 경우
        decision = str(raw_content)

    # 공백 제거 및 소문자 변환으로 파싱 안정성 향상
    decision = decision.strip().lower()
    print(f"   -> LLM 판단 결과: {decision}")

    # 분류 결과를 State에 저장
    if "escalate" in decision:
        return {"next_step": "escalate"}    # 상담원 이관 트랙
    else:
        return {"next_step": "consultant"}  # 자동 상담 트랙

In [180]:
# ============================================================
# [Node 2] consultant_node - 상담 AI 노드 (Worker 역할)
#
# 역할: 도구(매뉴얼 검색)를 활용하여 고객 문의에 답변을 생성합니다.
# 도구 사용이 필요하면 tool_calls를 포함한 응답을 반환합니다.
# ============================================================
def consultant_node(state: AgentState):
    """매뉴얼 검색 도구를 사용하여 고객 문의에 답변하는 AI 상담 노드"""
    print("\n--- [2-A] 상담 AI 답변 생성 중 ---")
    
    # 도구가 연결된 모델로 현재 대화 기록 처리
    response = model_with_tools.invoke(state['messages'])
    
    return {'messages': [response]}

In [181]:
# ============================================================
# [Node 3] escalate_node - 상담원 이관 노드
#
# 역할: 분류 결과가 'escalate'인 경우 이관 메시지를 생성합니다.
# 실제 프로젝트에서는 CRM 티켓 생성, 슬랙 알림 등이 여기서 처리됩니다.
# ============================================================
from langchain_core.messages import AIMessage

def escalate_node(state: AgentState):
    """이메일을 전문 상담원에게 이관하고 안내 메시지를 생성합니다."""
    # 이관 완료 메시지를 대화 기록에 추가
    return {'messages': [AIMessage(content='해당 메일은 전문 상담원에게 이관되었습니다.')]}

In [182]:
# 도구 목록 (위에서 정의한 것과 동일)
tools = [search_manual]

In [183]:
# 등록된 도구 목록 확인
tools

In [184]:
# 도구 이름-함수 매핑 딕셔너리 생성
tools_by_name = {tool.name: tool for tool in tools}

In [185]:
# 매핑 딕셔너리 확인
tools_by_name

In [186]:
# ============================================================
# [Node 4] tool_node - 도구 실행 노드
#
# consultant_node가 요청한 도구를 실제로 실행합니다.
# 이전 예제의 tool_node와 동일한 로직이며, 재사용 가능합니다.
# ============================================================
from langchain.messages import ToolMessage

def tool_node(state: AgentState):
    """상담 AI가 요청한 도구를 실제로 실행하는 노드"""
    print("\n--- [Tool Node] 도구 직접 실행 ---")
    result = []
    
    # 가장 최근 메시지에서 도구 호출 요청 꺼내기
    last_message = state["messages"][-1]

    for tool_call in last_message.tool_calls:
        # 1단계: 도구 이름으로 실제 함수 찾기
        tool = tools_by_name[tool_call["name"]]

        # 2단계: 함수를 실행하여 결과를 얻기
        print(f"   -> 실행 중: {tool_call['name']}")
        tool_result = tool.invoke(tool_call["args"])

        # 3단계: 결과를 ToolMessage 형태로 포장 (tool_call_id 필수!)
        result.append(
            ToolMessage(
                content=str(tool_result),
                tool_call_id=tool_call["id"]
            )
        )

    # 실행 결과를 대화 기록에 추가
    return {"messages": result}

## Step 4. 그래프 생성

In [187]:
# ============================================================
# 전체 에이전트 그래프 구성
#
# 구조:
#   START → classify_node(LLM 분류)
#               ├─ 'escalate' → escalate_node → END
#               └─ 'consultant' → consultant_node
#                                   ├─ tool_calls 있음 → tool_node → consultant_node
#                                   └─ tool_calls 없음 → END
# ============================================================
from langgraph.graph import StateGraph, START, END

# 1. 워크플로우(그래프) 생성
agent_builder = StateGraph(AgentState)

# 2. 노드(작업자) 배치
agent_builder.add_node('classify_node', classify_node)     # LLM 분류 (Manager)
agent_builder.add_node('consultant_node', consultant_node) # AI 상담 (Worker)
agent_builder.add_node('escalate_node', escalate_node)     # 상담원 이관
agent_builder.add_node('tool_node', tool_node)             # 도구 실행

# 3. 시작 엣지: START → classify_node
agent_builder.add_edge(START, 'classify_node')

In [188]:
# ============================================================
# 라우터 1: 분류 결과에 따른 분기
#
# classify_node의 State를 읽어 다음 노드를 결정합니다.
# ============================================================
def route_after_classify(state: AgentState):
    """분류 결과에 따라 상담 또는 이관 경로를 선택합니다."""
    if state['next_step'] == 'escalate':
        return 'escalate'    # 이관 트랙
    else:
        return 'consultant'  # 자동 상담 트랙

In [189]:
# ============================================================
# 라우터 2: 도구 사용 여부 판단
#
# consultant_node 실행 후 도구 사용이 필요한지 판단합니다.
# (Part 1의 should_continue와 동일한 로직)
# ============================================================
def should_continue(state: AgentState):
    """상담 AI의 응답에 도구 호출이 필요한지 판단합니다."""
    last_message = state["messages"][-1]

    # LLM이 도구 호출(tool_calls)을 포함한 응답을 보낸 경우
    if last_message.tool_calls:
        return "tool_node"  # 도구 실행 후 다시 상담 노드로

    # 도구 호출이 없으면 종료 (최종 답변 완성)
    return END

In [190]:
# ============================================================
# 조건부 엣지 1: classify_node → (escalate_node | consultant_node)
#
# route_after_classify 함수가 'escalate' 또는 'consultant'를 반환하면
# 딕셔너리를 통해 해당 노드로 매핑됩니다.
# ============================================================
agent_builder.add_conditional_edges(
    'classify_node',       # 출발 노드
    route_after_classify,  # 분기 결정 함수
    {
        'escalate': 'escalate_node',      # 'escalate' → escalate_node
        'consultant': 'consultant_node'   # 'consultant' → consultant_node
    }
)

In [191]:
# ============================================================
# 조건부 엣지 2: consultant_node → (tool_node | END)
#
# 상담 AI가 도구를 사용해야 하면 tool_node로, 아니면 종료합니다.
# ============================================================
agent_builder.add_conditional_edges(
    'consultant_node',  # 상담 AI 노드 이후
    should_continue,    # 도구 사용 여부 판단
    ['tool_node', END]  # 가능한 목적지
)

In [192]:
# ============================================================
# 나머지 고정 엣지 연결
#
# tool_node 실행 후 → 다시 consultant_node (결과를 LLM에 전달)
# escalate_node 완료 후 → END
# ============================================================
agent_builder.add_edge('tool_node', 'consultant_node')  # 도구 결과 → 상담 AI에 전달
agent_builder.add_edge('escalate_node', END)            # 이관 완료 → 종료

In [193]:
# 최종 에이전트 컴파일
agent = agent_builder.compile()

In [194]:
# 완성된 에이전트 그래프 시각화 (복잡한 분기 구조 확인)
agent

## Step 5. 에이전트 실행 테스트

In [195]:
# ============================================================
# 테스트 1: 일반 문의 처리
#
# 예상 흐름:
#   [1] classify_node: LLM이 '비밀번호 변경' → 'consultant' 판단
#   [2] consultant_node: 도구 검색 필요 → tool_calls=[search_manual]
#   [Tool] tool_node: search_manual("비밀번호") → 위치 정보 반환
#   [3] consultant_node: 검색 결과로 최종 답변 생성 → END
# ============================================================
from langchain.messages import HumanMessage, SystemMessage

inputs = {"messages": [HumanMessage(content="비밀번호 변경은 어디서 해?")]}
response = agent.invoke(inputs)

In [196]:
# 전체 응답 확인
# - messages: 전체 대화 기록 (사용자 → 상담AI → 도구결과 → 최종답변)
# - next_step: 'consultant' (자동 상담으로 처리됨)
response

In [199]:
# 최종 답변만 추출
response["messages"][-1].content

In [202]:
# ============================================================
# 테스트 2: 환불 요청 처리 (이전 예제와 비교)
#
# '환급' 키워드도 LLM이 환불 요청으로 정확하게 분류합니다!
# → 규칙 기반의 한계를 LLM 기반으로 극복한 사례
#
# 예상 흐름:
#   [1] classify_node: LLM이 '당장 환급해줘' → 'escalate' 판단
#   escalate_node → 이관 메시지 생성 → END
# ============================================================
inputs = {"messages": [HumanMessage(content="당장 환급해줘!")]}
response = agent.invoke(inputs)

In [201]:
# 환불 요청 처리 결과 확인
# - next_step: 'escalate' (LLM이 이관 필요로 판단)
# - 마지막 메시지: '해당 메일은 전문 상담원에게 이관되었습니다.'
response

---
## 📚 State 설계 원칙

([공식 문서](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph#what-belongs-in-state) 참고)

좋은 State 설계를 위한 핵심 원칙:

1. **필요한 것만 포함** - 모든 노드가 공유하는 정보만 State에 넣습니다.
2. **누적 vs 덮어쓰기** - 메시지는 누적(`operator.add`), 단순 값은 덮어쓰기
3. **타입 명시** - TypedDict를 사용하여 각 필드의 타입을 명확히 합니다.
4. **라우팅 정보 포함** - `next_step` 같은 필드로 분기 제어를 단순화합니다.

---
## 🎯 핵심 정리

| 패턴 | 예제 | 특징 |
|---|---|---|
| **기본 ReAct** | Part 1 | LLM ↔ 도구 반복 실행 |
| **Router (규칙)** | Part 2 | 키워드 기반 빠른 분기 |
| **Router (LLM)** | Part 3 | LLM 판단 기반 유연한 분기 |